# Phase 1 — Data-Quality Register & Cold-Start Census (Step 1.7)

**Scope of this notebook** — the checklist from `solution_plan.md` Step 1.7:

| # | Question | Output |
| --- | --- | --- |
| 1 | Nulls in critical fields, across all 14 tables | DQ register: null rates, with a handling rule per field |
| 2 | Exact duplicate rows | DQ register: duplicate count per table |
| 3 | Referential orphans on every declared join | DQ register: measured integrity % |
| 4 | Impossible values (negative prices, inverted date ranges, out-of-range codes) | DQ register: violation counts |
| 5 | Timezone / date-format consistency | DQ register: format check |
| 6 | Cold-start census: screens with zero/sparse bookings, no POI, no ridership | Sizing for the Phase 6 fallback ladder |

**Exit criterion (from the plan):** `docs/data_dictionary.md#dq-register`
complete; cold-start counts drive Phase 6's fallback design.

**As-of date.** Reused unchanged from Step 1.4: `2026-08-19`.

## 0 · Environment

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _project_root() -> Path:
    """Locate the repository root whether the kernel starts in / or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "agentiq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project tree.")


ROOT = _project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

In [2]:
%load_ext autoreload
%autoreload 2

from agentiq.data import DataLake, ProjectPaths

PATHS = ProjectPaths(ROOT).ensure_dirs()
lake = DataLake(PATHS.raw_data, cache_dir=PATHS.cache)
PATHS

ProjectPaths(root=C:\Users\AmitK\Downloads\AI-hackathon-repo\agentiq)

In [3]:
TABLE_NAMES = (
    "cities", "locations", "zone_demographics", "points_of_interest", "events",
    "route_stops", "route_schedules", "ridership_actuals", "vehicles", "screens",
    "dim_slot", "client_facts", "bookings", "lost_leads",
)
tables = {name: lake[name] for name in TABLE_NAMES}
AS_OF = pd.Timestamp("2026-08-19")  # fixed in Step 1.4 — do not derive from wall clock
print(f"{len(tables)} tables loaded; as-of: {AS_OF.date()}")

14 tables loaded; as-of: 2026-08-19


## 1.7.1 · Nulls in critical fields

**Goal.** Every non-zero null rate across all 14 tables, so each one gets an
explicit handling rule rather than being discovered as a surprise downstream.

In [4]:
null_register_rows = []
for name, df in tables.items():
    null_pct = (df.isna().mean() * 100).round(2)
    for column, pct in null_pct.loc[null_pct > 0].items():
        null_register_rows.append({"table": name, "column": column, "null_pct": pct})

null_register = pd.DataFrame(null_register_rows).sort_values(["table", "null_pct"], ascending=[True, False])
null_register

,table,column,null_pct
0,events,poi_id,23.43
5,lost_leads,company_name_raw,55.66
7,lost_leads,client_target_price_per_slot_per_day,47.59
8,lost_leads,price_gap_pct,47.59
4,lost_leads,client_id,44.34
6,lost_leads,quoted_price_per_slot_per_day,36.62
2,screens,vehicle_id,76.57
1,screens,location_id,23.43
3,screens,position,12.54


**Read — every null here is meaningful, not missing data:**

| Table.Column | Null % | Why it's null | Handling rule |
| --- | --- | --- | --- |
| `events.poi_id` | 23.4% | Street events/parades have no host POI | Treat null as "anchor to `anchor_location_id` only", never impute a POI |
| `screens.vehicle_id` | 76.6% | Static screens have no vehicle | Null is the static/mobile discriminator itself — never impute |
| `screens.location_id` | 23.4% | Mobile screens have no fixed location | Same — `location_id` XOR `vehicle_id` is exact (verified in Step 1.3) |
| `screens.position` | 12.5% | `metro_rail_coach` interior panels have no mount face | Null is structural (§ confirmed in Step 1.4 §2.3), not a gap |
| `lost_leads.company_name_raw` | 55.7% | Populated only for the 643 leads with no `client_id` | The two columns are complementary, never both null (verified below) |
| `lost_leads.client_target_price_per_slot_per_day` | 47.6% | No counter-offer was ever given | Absence itself is evidence — exclude, don't impute, when fitting the price-gap curve |
| `lost_leads.price_gap_pct` | 47.6% | Derived from the field above; null wherever it is | Same rule |
| `lost_leads.client_id` | 44.3% | Lead came from a new prospect, not an existing account | Use `company_name_raw` as the identity fallback |
| `lost_leads.quoted_price_per_slot_per_day` | 36.6% | Lead died before a quote was ever sent (`initial_inquiry` stage) | Same rule as `client_target_price_per_slot_per_day` |

No table has an unexplained null. Every null-bearing column's missingness is
itself a fact about the row (mobile vs static, prospect vs account, how far a
lead got) and must be preserved, not filled.

In [5]:
# Verify the lost_leads identity-fallback claim: every row has exactly one of the two identifiers.
leads = tables["lost_leads"]
has_client = leads["client_id"].notna()
has_raw_name = leads["company_name_raw"].notna()
print("rows with both populated:", (has_client & has_raw_name).sum())
print("rows with neither populated:", (~has_client & ~has_raw_name).sum())
print("rows with exactly one populated:", (has_client ^ has_raw_name).sum(), "of", len(leads))

rows with both populated: 0
rows with neither populated: 0
rows with exactly one populated: 1450 of 1450


## 1.7.2 · Exact duplicate rows

In [6]:
dupe_counts = {name: int(df.duplicated().sum()) for name, df in tables.items()}
dupe_register = pd.Series(dupe_counts, name="exact_duplicate_rows")
dupe_register

cities                0
locations             0
zone_demographics     0
points_of_interest    0
events                0
route_stops           0
route_schedules       0
ridership_actuals     0
vehicles              0
screens               0
dim_slot              0
client_facts          0
bookings              0
lost_leads            0
Name: exact_duplicate_rows, dtype: int64

**Read.** Zero exact duplicate rows in all 14 tables. No de-duplication step
is required anywhere in the pipeline.

## 1.7.3 · Referential orphans on every declared join

**Goal.** Re-measure integrity on the joins Step 1.3 mapped, plus the joins
Step 1.6 newly relies on (POI/events to locations), so nothing downstream is
built on an assumed-clean foreign key.

In [7]:
cities, locations, zone_demo = tables["cities"], tables["locations"], tables["zone_demographics"]
poi, events = tables["points_of_interest"], tables["events"]
route_stops, vehicles = tables["route_stops"], tables["vehicles"]
screens, client_facts, bookings = tables["screens"], tables["client_facts"], tables["bookings"]

checks = [
    ("locations.city_id", locations["city_id"], cities["city_id"]),
    ("locations.zone_id", locations["zone_id"], zone_demo["zone_id"]),
    ("screens.city_id", screens["city_id"], cities["city_id"]),
    ("screens.location_id (non-null)", screens["location_id"].dropna(), locations["location_id"]),
    ("screens.vehicle_id (non-null)", screens["vehicle_id"].dropna(), vehicles["vehicle_id"]),
    ("bookings.screen_id", bookings["screen_id"], screens["screen_id"]),
    ("bookings.client_id", bookings["client_id"], client_facts["client_id"]),
    ("lost_leads.client_id (non-null)", leads["client_id"].dropna(), client_facts["client_id"]),
    ("lost_leads.anchor_screen_id (non-null)", leads["anchor_screen_id"].dropna(), screens["screen_id"]),
    ("poi.anchor_location_id", poi["anchor_location_id"], locations["location_id"]),
    ("events.poi_id (non-null)", events["poi_id"].dropna(), poi["poi_id"]),
    ("events.anchor_location_id", events["anchor_location_id"], locations["location_id"]),
    ("vehicles.corridor_id", vehicles["corridor_id"], route_stops["corridor_id"]),
    ("route_schedules.route_id", tables["route_schedules"]["route_id"], route_stops["route_id"]),
]

integrity_rows = []
for label, child, parent in checks:
    orphans = int((~child.isin(parent)).sum())
    integrity_pct = round(100 * (1 - orphans / len(child)), 2) if len(child) else float("nan")
    integrity_rows.append({"join": label, "rows": len(child), "orphans": orphans, "integrity_pct": integrity_pct})

integrity_register = pd.DataFrame(integrity_rows)
integrity_register

,join,rows,orphans,integrity_pct
0,locations.city_id,910,0,100.0
1,locations.zone_id,910,0,100.0
2,screens.city_id,11163,0,100.0
3,screens.location_id (non-null),8548,0,100.0
4,screens.vehicle_id (non-null),2615,0,100.0
5,bookings.screen_id,191109,0,100.0
6,bookings.client_id,191109,0,100.0
7,lost_leads.client_id (non-null),807,0,100.0
8,lost_leads.anchor_screen_id (non-null),1450,0,100.0
9,poi.anchor_location_id,1375,0,100.0


**Read.** All 14 measured joins resolve at 100.00% integrity, with zero
orphans anywhere — consistent with Step 1.3's join-graph findings and now
extended to the context tables (`points_of_interest`, `events`) Step 1.3 did
not cover. There is no orphan population anywhere in this dataset; the only
"unreachable by history" population is the cold-start census in §1.7.6
below, which is a *coverage* gap, not a *referential* one.

## 1.7.4 · Impossible or out-of-range values

**Goal.** Check every numeric/date invariant that *should* hold if the data
is internally consistent — inverted date ranges, out-of-bound codes,
non-positive prices, and a cross-column arithmetic check.

In [8]:
computed_duration = (bookings["end_date"] - bookings["start_date"]).dt.days + 1
duration_mismatch = int((computed_duration != bookings["duration_days"]).sum())

checks_1_7_4 = {
    "bookings: duration_days != (end_date - start_date + 1)": duration_mismatch,
    "bookings: end_date < start_date": int((bookings["end_date"] < bookings["start_date"]).sum()),
    "bookings: booked_date after start_date (booked after flight started)": int(
        (bookings["booked_date"] > bookings["start_date"]).sum()
    ),
    "bookings: slots_booked_per_day outside [1, 6]": int(
        (~bookings["slots_booked_per_day"].between(1, 6)).sum()
    ),
    "bookings: contracted_price_per_slot_per_day <= 0": int(
        (bookings["contracted_price_per_slot_per_day"] <= 0).sum()
    ),
    "bookings: negative line_item_value": int((bookings["line_item_value"] < 0).sum()),
    "lost_leads: lost_date < lead_date": int((leads["lost_date"] < leads["lead_date"]).sum()),
    "lost_leads: price_gap_pct outside [-1, 5] sanity range": int(
        (~leads["price_gap_pct"].dropna().between(-1, 5)).sum()
    ),
    "lost_leads: negotiation_rounds outside [0, 10]": int(
        (~leads["negotiation_rounds"].between(0, 10)).sum()
    ),
}
pd.Series(checks_1_7_4, name="violations")

bookings: duration_days != (end_date - start_date + 1)                  0
bookings: end_date < start_date                                         0
bookings: booked_date after start_date (booked after flight started)    0
bookings: slots_booked_per_day outside [1, 6]                           0
bookings: contracted_price_per_slot_per_day <= 0                        0
bookings: negative line_item_value                                      0
lost_leads: lost_date < lead_date                                       0
lost_leads: price_gap_pct outside [-1, 5] sanity range                  0
lost_leads: negotiation_rounds outside [0, 10]                          0
Name: violations, dtype: int64

In [9]:
# Cross-table consistency: do the four age bands in zone_demographics sum to ~100?
age_cols = ["pct_age_under_18", "pct_age_18_34", "pct_age_35_54", "pct_age_55_plus"]
age_sum = zone_demo[age_cols].sum(axis=1)
print("age-band sum per zone — describe():")
print(age_sum.describe())

# Cross-column consistency: km and mile distances on the same POI row should agree
# to the stated conversion factor.
implied_mi = poi["distance_to_location_km"] * 0.621371
km_mi_diff = (implied_mi - poi["distance_to_location_mi"]).abs()
print("\nkm-vs-mile consistency, |implied_mi - stated_mi| — describe():")
print(km_mi_diff.describe())

age-band sum per zone — describe():
count    3.000000e+01
mean     1.000000e+02
std      3.731953e-15
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
dtype: float64

km-vs-mile consistency, |implied_mi - stated_mi| — describe():
count    1375.000000
mean        0.000253
std         0.000143
min         0.000001
25%         0.000130
50%         0.000252
75%         0.000376
max         0.000500
dtype: float64


**Read.** Every check returns zero violations. `duration_days` is exactly
consistent with the date range on all 191,109 lines; no booking has an
inverted date range or a booking date after its own flight start; no price,
slot count, or negotiation count falls outside its valid range. Age bands sum
to exactly 100.0 on all 30 zones (std ≈ 3.7e-15, i.e. floating-point noise,
not a real deviation). The km/mile distance pair agrees to within 0.0005 —
rounding only. **This dataset has no data-quality defects of this kind.**
The DQ register's "handling rule" for this whole section is simply: none
needed, but the checks stay in place as regression guards for Phase 10's
reproducible-rebuild requirement.

## 1.7.5 · Timezone / date-format consistency

In [10]:
print("cities.timezone values:", cities["timezone"].tolist())

route_schedules = tables["route_schedules"]
print("\nroute_schedules.start_time sample:", route_schedules["start_time"].head(3).tolist())
print(
    "start_time always strict HH:MM (no seconds, no AM/PM):",
    route_schedules["start_time"].str.match(r"^\d{2}:\d{2}$").all(),
)

cities.timezone values: ['America/New_York', 'America/Chicago', 'America/Denver']

route_schedules.start_time sample: ['05:03', '05:47', '06:31']
start_time always strict HH:MM (no seconds, no AM/PM): True


In [11]:
date_dtypes = []
for name, df in tables.items():
    for column in df.columns:
        if "date" in column.lower():
            date_dtypes.append({"table": name, "column": column, "dtype": str(df[column].dtype)})
pd.DataFrame(date_dtypes)

,table,column,dtype
0,events,start_date,datetime64[ns]
1,events,end_date,datetime64[ns]
2,ridership_actuals,date,datetime64[ns]
3,client_facts,relationship_start_date,datetime64[ns]
4,bookings,start_date,datetime64[ns]
5,bookings,end_date,datetime64[ns]
6,bookings,booked_date,datetime64[ns]
7,lost_leads,lead_date,datetime64[ns]
8,lost_leads,lost_date,datetime64[ns]
9,lost_leads,requested_start_date,datetime64[ns]


**Read.** Each of the three cities has a distinct IANA timezone
(`America/New_York`, `America/Chicago`, `America/Denver`) but the raw CSVs
carry **no explicit UTC offset or timezone-qualified timestamp anywhere** —
every date/time field is a bare local value. This is consistent (the plan
already assumes local time for daypart/slot reasoning) but it is a stated
assumption, not a verified one: nothing in the data proves `start_time` for a
DAT-prefixed row is actually Denver-local rather than, say, uniformly
UTC-recorded. Treat "all times are local to their row's city" as a
documented assumption carried into Phase 3/4. Every date column parses
cleanly to `datetime64[ns]` with pandas' default parser and `start_time` is
uniformly `HH:MM`, so no format-consistency defects exist within a column.

## 1.7.6 · Cold-start census

**Goal.** Exactly how many screens have zero bookings, sparse bookings, no
nearby POI, or no ridership coverage — the population that sizes how visible
Phase 6's fallback ladder will be in the demo.

In [12]:
screens_all = screens.copy()
screen_booking_counts = bookings["screen_id"].value_counts()
screens_all["booking_count"] = screens_all["screen_id"].map(screen_booking_counts).fillna(0)
screens_all["has_booking"] = screens_all["booking_count"] > 0

no_booking = ~screens_all["has_booking"]
print(f"screens with zero bookings: {no_booking.sum():,} of {len(screens_all):,} ({no_booking.mean() * 100:.1f}%)")

sparse = screens_all["booking_count"].between(1, 4)
print(f"screens with sparse bookings (1-4 lines): {sparse.sum():,} ({sparse.mean() * 100:.1f}%)")

screens with zero bookings: 1,224 of 11,163 (11.0%)
screens with sparse bookings (1-4 lines): 144 (1.3%)


In [13]:
# POI coverage — only meaningful for static screens, which are the only ones with a location_id.
locations_with_poi = set(poi["anchor_location_id"].unique())
screens_all["has_nearby_poi"] = screens_all["location_id"].isin(locations_with_poi)

static_screens = screens_all.loc[screens_all["location_id"].notna()]
no_poi = ~static_screens["has_nearby_poi"]
print(f"static screens with no POI at their location: {no_poi.sum():,} of {len(static_screens):,} ({no_poi.mean() * 100:.1f}%)")

static screens with no POI at their location: 0 of 8,548 (0.0%)


In [14]:
# Ridership coverage — only meaningful for mobile screens, via their vehicle's corridor.
corridors_with_schedule = set(route_schedules["corridor_id"].unique())
screens_all["corridor_id"] = screens_all["vehicle_id"].map(vehicles.set_index("vehicle_id")["corridor_id"])
screens_all["has_ridership"] = screens_all["corridor_id"].isin(corridors_with_schedule)

mobile_screens = screens_all.loc[screens_all["vehicle_id"].notna()]
no_ridership = ~mobile_screens["has_ridership"]
print(
    f"mobile screens with no ridership coverage: {no_ridership.sum():,} of {len(mobile_screens):,} "
    f"({no_ridership.mean() * 100:.1f}%)"
)

mobile screens with no ridership coverage: 0 of 2,615 (0.0%)


**Read.** Every static screen has at least one POI within range, and every
mobile screen's corridor has ridership coverage — the *only* signal gap in
this dataset is booking history. Cold-start is therefore a pure
**commercial-history** problem, not a context/audience-data problem: D1's
exposure models can run on any screen regardless of booking history, but D3's
pricing model cannot, which is exactly why the plan scopes the fallback
ladder to Phase 6.

In [15]:
print("cold-start (zero bookings) by city:")
print((screens_all.groupby("city_id", observed=True)["has_booking"].apply(lambda s: (~s).mean() * 100)).round(1))

print("\ncold-start (zero bookings) by screen_type:")
print((screens_all.groupby("screen_type", observed=True)["has_booking"].apply(lambda s: (~s).mean() * 100)).round(1))

cold-start (zero bookings) by city:


city_id
ACS    53.9
DAT     9.2
LH      0.0
Name: has_booking, dtype: float64

cold-start (zero bookings) by screen_type:
screen_type
bus                 26.5
bus_stop            37.5
metro_rail_coach     0.0
metro_station        1.5
Name: has_booking, dtype: float64


**Read — this re-confirms and sharpens Step 1.4 §4.4.** 11.0% of all screens
(1,224 of 11,163) have zero booking history network-wide, plus a further 1.3%
(144 screens) with only 1–4 lines — thin enough that a per-screen model would
still be unreliable. The gap is entirely concentrated: **53.9% of ACS
screens** and only 9.2% of DAT, 0.0% of LH. By type it concentrates in the
cheaper static formats — `bus_stop` (37.5%) and `bus` (26.5%) — while
`metro_station` (1.5%) and `metro_rail_coach` (0.0%) are essentially fully
covered. **The cold-start population is exactly the value-tier city's cheaper
static inventory** — a coherent, addressable population for the fallback
ladder's cohort-based rungs (peer screens, then zone × type × position × size
cohort), not a scattered set of one-off gaps.

## Carry-forward

| Output | Consumed by |
| --- | --- |
| Null register with per-field handling rules (§1.7.1) | Every engine that reads a nullable column |
| Zero exact duplicates confirmed (§1.7.2) | No de-dup step needed anywhere |
| 100% integrity on all 14 measured joins incl. POI/events (§1.7.3) | Confidence to join without defensive orphan-filtering code |
| Zero impossible-value violations (§1.7.4) | Regression guards for Phase 10's reproducible rebuild |
| Local-time assumption flagged as unverified (§1.7.5) | Phase 2 domain model — document the assumption explicitly |
| Cold-start is booking-history-only, not context-data (§1.7.6) | Scopes Phase 6's fallback ladder correctly |
| Cold-start census by city/type: 11.0% network-wide, concentrated in ACS + cheap static formats (§1.7.6) | Phase 6 fallback-ladder design, new-city scaling story |

**Next — Step 1.8 (parse the campaign briefs), then Step 1.9 (findings review
gate — do not start Phase 3 before this).**